In [1]:
!pip install pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 22.2 MB/s eta 0:00:00


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.base import clone
from sklearn.model_selection import KFold, StratifiedKFold
from pytabkit import RealMLP_TD_Classifier, TabM_D_Classifier
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv
/kaggle/input/heartdisease/Heart_Disease_Prediction.csv


In [3]:
class config:
    SEED = 42
    N_FOLDS = 5
    TARGET = 'Heart Disease'
    
    INPUT_DIR = '/kaggle/input/playground-series-s6e2'

CONFIG = config()

In [4]:
train = pd.read_csv(f'{CONFIG.INPUT_DIR}/train.csv')
train['source'] = 'train'
test = pd.read_csv(f'{CONFIG.INPUT_DIR}/test.csv')
test['source'] = 'test'
sample_sub = pd.read_csv(f'{CONFIG.INPUT_DIR}/sample_submission.csv')

org = pd.read_csv('/kaggle/input/heartdisease/Heart_Disease_Prediction.csv')
org['source'] = 'original'

In [5]:
for df, name in zip([train, test, org], ['train', 'test', 'original']):
    print(f'NULL VALUE COUNTS FOR {name}:')
    print(df.isnull().sum())
    print('='*30)
    print(f'{name} shape:')
    print(df.shape)
    print('='*30)
    if name == 'train':
        print('General EDA -- TRAIN ONLY', end='\n')
        print(f'Dtypes :', end='\n')
        print(df.dtypes)
        print('='*30)
        print(f'NUMBER OF UNIQUE VALUES :', end='\n')
        print(df.nunique())
        print('='*30)
    

NULL VALUE COUNTS FOR train:
id                         0
Age                        0
Sex                        0
Chest pain type            0
BP                         0
Cholesterol                0
FBS over 120               0
EKG results                0
Max HR                     0
Exercise angina            0
ST depression              0
Slope of ST                0
Number of vessels fluro    0
Thallium                   0
Heart Disease              0
source                     0
dtype: int64
train shape:
(630000, 16)
General EDA -- TRAIN ONLY
Dtypes :
id                           int64
Age                          int64
Sex                          int64
Chest pain type              int64
BP                           int64
Cholesterol                  int64
FBS over 120                 int64
EKG results                  int64
Max HR                       int64
Exercise angina              int64
ST depression              float64
Slope of ST                  int64
Number of ves

In [6]:
combine = pd.concat([train.drop(columns='id'), test.drop(columns='id'), org], ignore_index=True).reset_index()
combine.shape

(900270, 16)

In [7]:
train['Heart Disease'].unique()

array(['Presence', 'Absence'], dtype=object)

In [8]:
class_mapping = {
    'Presence': 1,
    'Absence': 0
}
rev_class_mapping = {
    0: 'Absence',
    1: 'Presence'
}

FEATURES = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source']]
X = train[FEATURES]
X_org = org[FEATURES]

y = train[CONFIG.TARGET].map(class_mapping)
y_org = org[CONFIG.TARGET].map(class_mapping)

X_test = test[FEATURES]

In [9]:
xgb_params = {
    'n_estimators': 10000,
    'learning_rate': 0.005,
    'subsample': 0.8,
    # 'colsample_by_tree': 0.7,
    # 'sampling_method': 'gradient_based',
    # 'reg_alpha': 2.0,
    # 'reg_lambda': 4.0,
    'eval_metric': 'auc',
    'early_stopping_rounds': 100,
    'random_state': CONFIG.SEED
}

lgb_params = {
    'n_estimators': 10_000,
    'learning_rate': 0.005,
    'max_depth': 8,                 # same philosophy
    'num_leaves': 2 ** 8,           # typical rule: ≤ 2^max_depth
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    # 'reg_lambda': 4.0,
    'random_state': CONFIG.SEED,
    'early_stopping_rounds': 100,
    'eval_metric': 'auc',
    'n_jobs': -1,
    # 'verbose': 200,
    'verbosity':-1,
    # 'device': 'cuda'
    # 'cat_feature': CATS,            # pass categorical indices/names
}

cat_params = {
    'iterations': 10_000,          # same as n_estimators
    'learning_rate': 0.005,
    'depth': 8,                    # max_depth equivalent
    'subsample': 0.8,              # bagging
    'colsample_bylevel': 0.7,      # feature fraction per split
    'reg_lambda': 4.0,             # L2
    'random_seed': CONFIG.SEED,
    'early_stopping_rounds': 100,
    'eval_metric': 'AUC',
    'thread_count': -1,            # use all cores
    'verbose': 200,                # same as XGB verbose
    # 'verbosity':-1
    # 'cat_features': CATS,          # list of column names / indices
}

real_mlp_params = {
        'device': 'cuda',
        'random_state': 42,
        'verbosity': 2,
        'val_metric_name': '1-auc_ovo',
        'n_epochs': 60,
        'batch_size': 1024,
        'n_ens': 8,
        'use_early_stopping': True,
        'early_stopping_additive_patience': 20,
        'early_stopping_multiplicative_patience': 1,
        'act': "mish",
        'embedding_size': 8,
        'first_layer_lr_factor': 0.5962121993798933,
        'hidden_sizes': "rectangular",
        'hidden_width': 384,
        'lr': 0.01,
        'ls_eps': 0.0,
        'ls_eps_sched': "coslog4",
        'max_one_hot_cat_size': 18,
        'n_hidden_layers': 3,
        'p_drop': 0.1,
        'p_drop_sched': "flat_cos",
        'plr_hidden_1': 16,
        'plr_hidden_2': 8,
        'plr_lr_factor': 0.1151437622270563,
        'plr_sigma': 2.3316811282666916,
        'scale_lr_factor': 2.244801835541429,
        'sq_mom': 1.0 - 0.011834054955582318,
        'wd': 0.02369230879235962,
    }

In [10]:
skf = StratifiedKFold(n_splits=CONFIG.N_FOLDS, random_state=CONFIG.SEED, shuffle=True)
kf = KFold(n_splits=CONFIG.N_FOLDS, random_state=CONFIG.SEED, shuffle=True)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for i, (train_idx, val_idx) in enumerate(skf.split(X, y), 0):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    # X_org_n = X_org.copy()
    # y_org_n = y_org.copy()
    # X_org_n = pd.concat([X_org_n]*20, axis=0, ignore_index=True)
    # y_org_n = pd.concat([y_org_n]*20, axis=0, ignore_index=True)
    # X_train = pd.concat([X_train, X_org_n], axis=0, ignore_index=True)
    # y_train = pd.concat([y_train, y_org_n], axis=0, ignore_index=True)
    # model = clone(xgb.XGBClassifier(**xgb_params))
    # model = clone(cb.CatBoostClassifier(**cat_params))
    # model = clone(lgb.LGBMClassifier(**lgb_params))

    model = clone(RealMLP_TD_Classifier(**real_mlp_params))
    model.fit(X_train, y_train,
             X_val, y_val, 
             )
    preds = model.predict_proba(X_val)[:,1]
    oof_preds[val_idx] = preds
    test_preds += (model.predict_proba(X_test)[:, 1] / CONFIG.N_FOLDS)
    print(f'SCORE FOR FOLD{i} : {roc_auc_score(y_val, preds)}')
    
overall_roc_auc = roc_auc_score(y, oof_preds)
print(f'SCORE ACROSS ALL FOLDS : {overall_roc_auc}')

Columns classified as continuous: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
Columns classified as categorical: []


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 1/60: val 1-auc_ovo = 0.045974
Epoch 2/60: val 1-auc_ovo = 0.045352
Epoch 3/60: val 1-auc_ovo = 0.045156
Epoch 4/60: val 1-auc_ovo = 0.045103
Epoch 5/60: val 1-auc_ovo = 0.045097
Epoch 6/60: val 1-auc_ovo = 0.045112
Epoch 7/60: val 1-auc_ovo = 0.045014
Epoch 8/60: val 1-auc_ovo = 0.045019
Epoch 9/60: val 1-auc_ovo = 0.044892
Epoch 10/60: val 1-auc_ovo = 0.044831
Epoch 11/60: val 1-auc_ovo = 0.044797
Epoch 12/60: val 1-auc_ovo = 0.044785
Epoch 13/60: val 1-auc_ovo = 0.044772
Epoch 14/60: val 1-auc_ovo = 0.044796
Epoch 15/60: val 1-auc_ovo = 0.044778
Epoch 16/60: val 1-auc_ovo = 0.044796
Epoch 17/60: val 1-auc_ovo = 0.044765
Epoch 18/60: val 1-auc_ovo = 0.044724
Epoch 19/60: val 1-auc_ovo = 0.044669
Epoch 20/60: val 1-auc_ovo = 0.044578
Epoch 21/60: val 1-auc_ovo = 0.044542
Epoch 22/60: val 1-auc_ovo = 0.044529
Epoch 23/60: val 1-auc_ovo = 0.044504
Epoch 24/60: val 1-auc_ovo = 0.044437
Epoch 25/60: val 1-auc_ovo = 0.044426
Epoch 26/60: val 1-auc_ovo = 0.044401
Epoch 27/60: val 1-au

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


SCORE FOR FOLD0 : 0.9556118981337355
Columns classified as continuous: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
Columns classified as categorical: []


...

SCORE FOR FOLD3 : 0.9549607286578142
Columns classified as continuous: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
Columns classified as categorical: []


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 1/60: val 1-auc_ovo = 0.045851
Epoch 2/60: val 1-auc_ovo = 0.045330
Epoch 3/60: val 1-auc_ovo = 0.045027
Epoch 4/60: val 1-auc_ovo = 0.044962
Epoch 5/60: val 1-auc_ovo = 0.044939
Epoch 6/60: val 1-auc_ovo = 0.044942
Epoch 7/60: val 1-auc_ovo = 0.044876
Epoch 8/60: val 1-auc_ovo = 0.044837
Epoch 9/60: val 1-auc_ovo = 0.044719
Epoch 10/60: val 1-auc_ovo = 0.044639
Epoch 11/60: val 1-auc_ovo = 0.044611
Epoch 12/60: val 1-auc_ovo = 0.044599
Epoch 13/60: val 1-auc_ovo = 0.044595
Epoch 14/60: val 1-auc_ovo = 0.044589
Epoch 15/60: val 1-auc_ovo = 0.044642
Epoch 16/60: val 1-auc_ovo = 0.044640
Epoch 17/60: val 1-auc_ovo = 0.044619
Epoch 18/60: val 1-auc_ovo = 0.044577
Epoch 19/60: val 1-auc_ovo = 0.044568
Epoch 20/60: val 1-auc_ovo = 0.044541
Epoch 21/60: val 1-auc_ovo = 0.044438
Epoch 22/60: val 1-auc_ovo = 0.044434
Epoch 23/60: val 1-auc_ovo = 0.044382
Epoch 24/60: val 1-auc_ovo = 0.044325
Epoch 25/60: val 1-auc_ovo = 0.044319
Epoch 26/60: val 1-auc_ovo = 0.044297
Epoch 27/60: val 1-au

`Trainer.fit` stopped: `max_epochs=60` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


SCORE FOR FOLD4 : 0.9557628391788062
SCORE ACROSS ALL FOLDS : 0.9552061200943807


In [11]:
sample_sub['Heart Disease'] = test_preds
sample_sub.to_csv(f'submission_{overall_roc_auc}.csv', index=False)